# Manuscript-Style Result Figures

Editable notebook version of `scripts/figures/plot_manuscript_style_result_figures.py`.

Recommended manual tuning points:
- `MODEL_DISPLAY`, `MODEL_COLORS`, and `SPECIES_COLORS`
- `HEATMAP_FEATURES` and `HEATMAP_LABELS`
- panel titles, figure sizes, and annotation density
- manuscript output directory and file names
- integrated explanation architecture figure and its module/rank summaries


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from adjustText import adjust_text
from sklearn.metrics import balanced_accuracy_score

NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATES = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(path for path in CANDIDATES if (path / "data" / "processed" / "final_sequence_dataset.tsv").exists())

COMPARE_SUITE_DIR = REPO_ROOT / "results" / "modeling" / "figure1_compare_latest_winners"
FOCAL_SUITE_DIR = REPO_ROOT / "results" / "modeling" / "final_repeated_stratified_suite"
FIG4_SUITE_DIR = REPO_ROOT / "results" / "modeling" / "final_within_species_suite"
SPECIES_SUITE_DIR = REPO_ROOT / "results" / "modeling" / "final_species_suite"
WITHIN_SUITE_DIR = REPO_ROOT / "results" / "modeling" / "final_within_species_suite"
PROMOTED_SUITE_DIR = REPO_ROOT / "results" / "modeling" / "final_within_species_suite"
HIER_DIR = REPO_ROOT / "results" / "modeling" / "final_hierarchical_block_comparison"
DATASET = REPO_ROOT / "data" / "processed" / "final_sequence_dataset.tsv"
OUTPUT_DIR = HIER_DIR / "figures" / "manuscript_utr3_integrated"
FIG3_DATA_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_SAVE_PATH = REPO_ROOT / "notebooks" / "figures"

print(REPO_ROOT)
print(OUTPUT_DIR)


In [ ]:
SCHEME_LABELS = {
    "grouped_species": "Leave-one-species-out",
    "grouped_within_species_clade": "Leave-one-lineage-out",
    "repeated_stratified": "Repeated stratified CV",
}

FEATURE_SET_DISPLAY = {
    "species_only": "Species only",
    "ecology_only": "Ecology only",
    "molecular_only": "Molecular only",
    "ecology_molecular": "Ecology + molecular",
    "integrated_taxon_mechanism": "Integrated",
    "ecology_molecular_lineage_core": "Lineage core",
    "ecology_molecular_lineage_plus_context": "Lineage + context",
    "ecology_molecular_lineage_plus_species_summaries": "Lineage + species summaries",
}

MODEL_NAME_DISPLAY = {
    "random_forest_shallow": "Random forest",
    "lightgbm": "LightGBM",
    "xgboost": "XGBoost",
    "catboost": "CatBoost",
    "gradient_boosting": "Gradient boosting",
    "penalized_logistic_l2": "L2 logistic",
    "penalized_logistic_elasticnet": "Elastic-net logistic",
}

MODEL_DISPLAY = {
    ("ecology_molecular_lineage_plus_context", "lightgbm"): "Lineage + context\nLightGBM",
    ("species_only", "random_forest_shallow"): "Species only\nRandom forest",
    ("ecology_molecular", "xgboost"): "Ecology + molecular\nXGBoost",
}

MODEL_COLORS = {
    ("ecology_molecular_lineage_plus_context", "lightgbm"): "#9fc5d5",
    ("species_only", "random_forest_shallow"): "#c7d7a1",
    ("ecology_molecular", "xgboost"): "#efc3a4",
}

SPECIES_COLORS = {
    "orthohantavirus_andesense": "#1b9e77",
    "orthohantavirus_dobravaense": "#d95f02",
    "orthohantavirus_hantanense": "#7570b3",
    "orthohantavirus_puumalaense": "#66a61e",
    "orthohantavirus_seoulense": "#e7298a",
    "orthohantavirus_sinnombreense": "#a6761d",
    "orthohantavirus_tulaense": "#1f78b4",
}

TERM_DISPLAY = {
    "hosttrait_combine_adult_mass_g_z": "Host body mass",
    "hosttrait_combine_litter_size_n_z": "Host litter size",
    "hosttrait_combine_gestation_length_d_z": "Host gestation length",
    "hosttrait_combine_dispersal_km_z": "Host dispersal",
    "species_feat_hostdiv_faith_pd_z": "Host Faith's PD",
    "overlap_opportunity_local_z": "Overlap opportunity",
    "arha_prev_local_z": "Local host prevalence",
    "overlap_opportunity_local_z:species_feat_hostdiv_faith_pd_z": "Overlap x host PD",
    "species_feat_mean_cramers_v_overall_z": "LD burden",
    "species_feat_prop_significant_overall_z": "LD significant-pair fraction",
    "species_feat_utr5_S_structdist_mean_z": "UTR5 S structure distance",
    "species_feat_utr5_M_structdist_mean_z": "UTR5 M structure distance",
    "species_feat_utr5_L_structdist_mean_z": "UTR5 L structure distance",
    "species_feat_utr3_S_structdist_mean_z": "UTR3 S structure distance",
    "species_feat_utr3_M_structdist_mean_z": "UTR3 M structure distance",
    "species_feat_utr3_L_structdist_mean_z": "UTR3 L structure distance",
    "tree_discordance_score_z": "Tree discordance score",
}

HEATMAP_LABELS = {
    "observed_rate": "Observed rate",
    "overlap_opportunity_local": "Overlap",
    "arha_prev_local": "Local prevalence",
    "species_feat_hostdiv_faith_pd": "Faith's PD",
    "species_feat_mean_cramers_v_overall": "LD burden",
    "species_feat_prop_significant_overall": "LD sig. frac.",
    "species_feat_utr5_S_structdist_mean": "UTR5 S struct.",
    "species_feat_utr5_M_structdist_mean": "UTR5 M struct.",
    "species_feat_utr5_L_structdist_mean": "UTR5 L struct.",
    "species_feat_utr3_S_structdist_mean": "UTR3 S struct.",
    "species_feat_utr3_M_structdist_mean": "UTR3 M struct.",
    "species_feat_utr3_L_structdist_mean": "UTR3 L struct.",
    "tree_discordance_score": "Tree discord.",
}

BLOCK_DISPLAY = {
    "ecology_interaction_block": "Ecology +\ninteraction",
    "molecular_plus_utr3_block": "Molecular\n+ UTR3",
    "integrated_plus_utr3_block": "Integrated\n+ UTR3",
    "integrated_tree_sensitivity": "Integrated\n+ tree",
    "species_only": "Species only",
    "integrated_block": "Integrated",
}

FAMILY_COLORS = {
    "Species identity": "#6c8ebf",
    "Host life history": "#4daf8b",
    "Host phylogenetic diversity": "#88c999",
    "Overlap opportunity": "#e3b34c",
    "Local host prevalence": "#d9a441",
    "LD / coadaptation": "#d95f5f",
    "UTR structure": "#9b6fb6",
    "Tree incongruence": "#708090",
    "Geographic / sampling context": "#7fa4b7",
    "Controls / sampling": "#b7b7b7",
}

SPECIES_LABELS = {
    "orthohantavirus_andesense": "ANDV",
    "orthohantavirus_dobravaense": "DOBV",
    "orthohantavirus_hantanense": "HTNV",
    "orthohantavirus_puumalaense": "PUUV",
    "orthohantavirus_seoulense": "SEOV",
    "orthohantavirus_sinnombreense": "SNV",
    "orthohantavirus_tulaense": "TULV",
}

def set_style():
    mpl.rcParams.update(
        {
            "font.family": "Helvetica Neue",
            "font.size": 7,
            "axes.titlesize": 7.5,
            "axes.labelsize": 7,
            "xtick.labelsize": 6.5,
            "ytick.labelsize": 6.5,
            "legend.fontsize": 6.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
            "savefig.facecolor": "white",
            "axes.linewidth": 0.7,
            "xtick.major.width": 0.7,
            "ytick.major.width": 0.7,
            "xtick.major.size": 2.5,
            "ytick.major.size": 2.5,
            "lines.linewidth": 1.0,
        }
    )

def species_short(name):
    return SPECIES_LABELS.get(name, name)

def lighten_color(color, amount=0.45):
    rgb = np.array(mpl.colors.to_rgb(color))
    return tuple(rgb + (1.0 - rgb) * amount)

def panel_label(ax, label, x=-0.10, y=1.03):
    ax.text(x, y, label.lower(), transform=ax.transAxes, fontsize=10, fontweight="bold", va="top")

def pretty_feature_set(name):
    if name in FEATURE_SET_DISPLAY:
        return FEATURE_SET_DISPLAY[name]
    return str(name).replace("_", " ").replace("taxon mechanism", "integrated").title()

def pretty_model_name(name):
    if name in MODEL_NAME_DISPLAY:
        return MODEL_NAME_DISPLAY[name]
    return str(name).replace("_", " ").title()

def model_pair_label(feature_set, model):
    return f"{pretty_feature_set(feature_set)}\n{pretty_model_name(model)}"

def pretty_term(term):
    if term in TERM_DISPLAY:
        return TERM_DISPLAY[term]
    out = str(term)
    out = out.replace("_z", "")
    out = out.replace("species_feat_", "")
    out = out.replace("hosttrait_combine_", "host ")
    out = out.replace("_structdist_mean", " structure distance")
    out = out.replace("_", " ")
    return out.title()

def ordered_present(present, preferred):
    seen = set()
    out = []
    for item in preferred:
        if item in present and item not in seen:
            out.append(item)
            seen.add(item)
    for item in present:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out

set_style()


In [ ]:
def make_figure1(compare_suite_dir=COMPARE_SUITE_DIR, repeat_suite_dir=FOCAL_SUITE_DIR, hier_dir=HIER_DIR, outpath=None):
    outpath = outpath or FIGURE_SAVE_PATH / "figure1_model.png"
    fold_df = pd.read_csv(compare_suite_dir / "cv_fold_metrics.tsv", sep="\t")
    summary_df = pd.read_csv(compare_suite_dir / "summary.tsv", sep="\t")
    repeat_pred_df = pd.read_csv(repeat_suite_dir / "repeated_stratified_cv_predictions.tsv", sep="\t")

    ppc_candidates = [
        hier_dir / "integrated_plus_utr3_block" / "ppc_by_species.tsv",
        hier_dir / "integrated_block" / "ppc_by_species.tsv",
    ]
    ppc_path = next((path for path in ppc_candidates if path.exists()), None)
    if ppc_path is None:
        raise FileNotFoundError("No species posterior-predictive table found in hierarchical output")
    ppc_df = pd.read_csv(ppc_path, sep="\t")

    scheme_order = ["repeated_stratified", "grouped_species", "grouped_within_species_clade"]
    suite_order = ["ecology_only", "ecology_molecular", "integrated_taxon_mechanism"]
    suite_labels = {
        "ecology_only": "Ecology",
        "ecology_molecular": "Ecology + molecular",
        "integrated_taxon_mechanism": "Integrated",
    }
    model_order = ["catboost", "lightgbm", "random_forest_shallow", "xgboost"]
    model_short = {
        "catboost": "CB",
        "lightgbm": "LGBM",
        "random_forest_shallow": "RF",
        "xgboost": "XGB",
    }
    model_colors = {
        "catboost": "#e76f51",
        "lightgbm": "#1f4e79",
        "random_forest_shallow": "#2a9d8f",
        "xgboost": "#e9c46a",
    }

    compare_pairs = [(suite, model) for suite in suite_order for model in model_order]
    fold_df = fold_df[
        fold_df["feature_set"].isin(suite_order) & fold_df["model"].isin(model_order) & fold_df["cv_scheme"].isin(scheme_order)
    ].copy()
    summary_df = summary_df[
        summary_df["feature_set"].isin(suite_order) & summary_df["model"].isin(model_order) & summary_df["cv_scheme"].isin(scheme_order)
    ].copy()

    fig = plt.figure(figsize=(13.2, 7.8), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.14, 1.12])
    axA = fig.add_subplot(gs[0, :])
    axB = fig.add_subplot(gs[1, 0])
    axC = fig.add_subplot(gs[1, 1])
    axD = fig.add_subplot(gs[1, 2])

    panel_label(axA, "A")
    panel_label(axB, "B", y=1.09)
    panel_label(axC, "C", y=1.09)
    panel_label(axD, "D", y=1.09)
    for ax in (axB, axC, axD):
        ax.set_box_aspect(0.95)

    repeat_plot = (
        repeat_pred_df.groupby(["row_index", "tip_name", "species", "target"], as_index=False)["probability"]
        .mean()
        .sort_values("probability", ascending=False)
        .reset_index(drop=True)
    )
    prevalence = float(repeat_plot["target"].mean())

    positions = []
    xticklabels = []
    cursor = 0.0
    scheme_spans = []
    suite_spans = []
    for scheme in scheme_order:
        scheme_start = cursor - 0.45
        for suite in suite_order:
            suite_start = cursor - 0.35
            for model in model_order:
                pos = cursor
                positions.append(pos)
                xticklabels.append(model_short[model])
                sub = fold_df[
                    (fold_df["cv_scheme"] == scheme)
                    & (fold_df["feature_set"] == suite)
                    & (fold_df["model"] == model)
                ].copy()
                vals = sub["average_precision"].dropna().to_numpy()
                if len(vals):
                    vp = axA.violinplot(vals, positions=[pos], widths=0.78, showextrema=False)
                    for body in vp["bodies"]:
                        body.set_facecolor(model_colors[model])
                        body.set_edgecolor("none")
                        body.set_alpha(0.26)
                    bp = axA.boxplot(
                        vals,
                        positions=[pos],
                        widths=0.34,
                        patch_artist=True,
                        showfliers=True,
                        medianprops={"color": "#3b3b3b", "linewidth": 1.0},
                        whiskerprops={"color": "#555555", "linewidth": 0.8},
                        capprops={"color": "#555555", "linewidth": 0.8},
                        boxprops={"facecolor": "#f7f7f7", "edgecolor": "#555555", "linewidth": 0.8},
                        flierprops={"marker": "o", "markersize": 2.3, "markerfacecolor": "#555555", "markeredgecolor": "#555555", "alpha": 0.5},
                    )
                    for patch in bp["boxes"]:
                        patch.set_alpha(0.92)
                    jitter = np.linspace(-0.11, 0.11, len(vals)) if len(vals) > 1 else np.array([0.0])
                    axA.scatter(
                        np.full(len(vals), pos) + jitter,
                        vals,
                        s=13,
                        color=model_colors[model],
                        alpha=0.62,
                        edgecolor="white",
                        linewidth=0.28,
                        zorder=3,
                    )
                stat = summary_df[
                    (summary_df["cv_scheme"] == scheme)
                    & (summary_df["feature_set"] == suite)
                    & (summary_df["model"] == model)
                ]
                if not stat.empty:
                    axA.scatter(
                        [pos],
                        [float(stat["oof_average_precision"].iloc[0])],
                        marker="D",
                        s=18,
                        color="#202020",
                        edgecolor="white",
                        linewidth=0.35,
                        zorder=4,
                    )
                cursor += 0.82
            suite_end = cursor - 0.47
            suite_spans.append((scheme, suite, suite_start, suite_end))
            cursor += 0.45
        scheme_end = cursor - 0.62
        scheme_spans.append((scheme, scheme_start, scheme_end))
        cursor += 0.95

    for idx, (scheme, start, end) in enumerate(scheme_spans):
        axA.axvspan(start, end, color="#fafaf8" if idx % 2 == 0 else "#ffffff", zorder=0)
        axA.text(
            (start + end) / 2,
            0.985,
            SCHEME_LABELS[scheme],
            transform=axA.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=7.2,
            color="#4d4d4d",
        )
    for _, suite, start, end in suite_spans:
        axA.text(
            (start + end) / 2,
            0.935,
            suite_labels[suite],
            transform=axA.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=6.1,
            color="#6a6a6a",
        )

    axA.axhline(prevalence, color="#555555", linestyle=(0, (4, 4)), linewidth=1.2)
    axA.text(max(positions) + 0.22, prevalence + 0.006, "prevalence", fontsize=6.1, color="#555555")
    axA.set_xlim(min(positions) - 0.55, max(positions) + 0.7)
    axA.set_ylim(0.0, 1.0)
    axA.set_ylabel("Average precision")
    axA.set_xticks(positions, xticklabels)
    axA.tick_params(axis="x", labelsize=5.8, pad=2)
    model_handles = [
        Line2D([0], [0], marker="o", linestyle="", markersize=5.5, markerfacecolor=model_colors[m], markeredgecolor=model_colors[m], label=pretty_model_name(m))
        for m in model_order
    ]
    model_handles.append(
        Line2D([0], [0], marker="D", linestyle="", markersize=4.8, markerfacecolor="#202020", markeredgecolor="#202020", label="OOF AP")
    )
    axA.legend(
        handles=model_handles,
        frameon=False,
        fontsize=6.1,
        ncol=5,
        loc="lower left",
        bbox_to_anchor=(0.0, 1.01, 1.0, 0.08),
        mode="expand",
        borderaxespad=0.0,
        handletextpad=0.3,
        columnspacing=0.8,
    )

    x = np.arange(len(repeat_plot))
    probs = repeat_plot["probability"].to_numpy()
    target = repeat_plot["target"].to_numpy()
    axB.fill_between(x, 0, probs, color="#fee8c8", alpha=0.8)
    axB.plot(x, probs, color="#2c3e50", linewidth=2)
    axB.axhline(prevalence, color="#555555", linestyle=(0, (4, 4)), linewidth=1.2)
    pos_idx = np.where(target == 1)[0]
    neg_idx = np.where(target == 0)[0]
    axB.scatter(pos_idx, np.full(pos_idx.shape, 1.01), marker="|", s=140, color="#bc4749", clip_on=False, transform=axB.get_xaxis_transform())
    neg_sub = neg_idx[:: max(1, len(neg_idx) // 120)]
    axB.scatter(neg_sub, np.full(len(neg_sub), -0.045), marker="|", s=50, color="#b8b8b8", clip_on=False, transform=axB.get_xaxis_transform())
    for i, species in enumerate(repeat_plot["species"]):
        axB.vlines(i, -0.12, -0.07, color=SPECIES_COLORS.get(species, "#777777"), linewidth=1)
    axB.set_ylim(-0.16, 1.05)
    axB.set_xlim(0, len(repeat_plot) - 1)
    axB.set_ylabel("Predicted probability")
    axB.set_xlabel("Repeated-stratified out-of-fold rank")
    axB.text(0.02, 0.98, "top ticks: reassortant", transform=axB.transAxes, va="top", fontsize=5.8, color="#555555")
    axB.text(0.02, 0.90, "bottom strip: species", transform=axB.transAxes, va="top", fontsize=5.8, color="#555555")

    threshold_grid = np.linspace(0.01, 0.99, 199)
    best_t = max(threshold_grid, key=lambda t: balanced_accuracy_score(repeat_plot["target"], (repeat_plot["probability"] >= t).astype(int)))
    repeat_plot["predicted"] = (repeat_plot["probability"] >= best_t).astype(int)
    actual_yes = int((repeat_plot["target"] == 1).sum())
    actual_no = int((repeat_plot["target"] == 0).sum())
    cm_cells = {
        (0, 0): int(((repeat_plot["predicted"] == 0) & (repeat_plot["target"] == 1)).sum()),
        (0, 1): int(((repeat_plot["predicted"] == 0) & (repeat_plot["target"] == 0)).sum()),
        (1, 0): int(((repeat_plot["predicted"] == 1) & (repeat_plot["target"] == 1)).sum()),
        (1, 1): int(((repeat_plot["predicted"] == 1) & (repeat_plot["target"] == 0)).sum()),
    }
    cm_mat = np.array([
        [cm_cells[(0, 0)] / actual_yes if actual_yes else 0.0, cm_cells[(0, 1)] / actual_no if actual_no else 0.0],
        [cm_cells[(1, 0)] / actual_yes if actual_yes else 0.0, cm_cells[(1, 1)] / actual_no if actual_no else 0.0],
    ])
    axC.imshow(cm_mat, cmap=mpl.cm.Greens, vmin=0, vmax=1)
    for spine in axC.spines.values():
        spine.set_linewidth(1.2)
        spine.set_color("#4a4a4a")
    axC.set_xticks([0, 1], ["Yes", "No"])
    axC.set_yticks([0, 1], ["No", "Yes"])
    axC.set_xlabel(f"Known reassortant\nthreshold = {best_t:.2f}")
    axC.set_ylabel("Predicted reassortant")
    axC.set_xticks(np.arange(-0.5, 2, 1), minor=True)
    axC.set_yticks(np.arange(-0.5, 2, 1), minor=True)
    axC.grid(which="minor", color="#c7c7c7", linestyle="-", linewidth=0.8)
    axC.tick_params(which="minor", bottom=False, left=False)
    for (r, c), count in cm_cells.items():
        total = actual_yes if c == 0 else actual_no
        txt_color = "white" if cm_mat[r, c] > 0.5 else "#4a4a4a"
        axC.text(c, r - 0.03, f"{count}/{total}", ha="center", va="center", fontsize=15, color=txt_color, fontweight=500)
        axC.text(c, r + 0.18, f"{count / total:.1%}", ha="center", va="center", fontsize=8.2, color=txt_color)

    plot_df = ppc_df.sort_values("observed_rate").reset_index(drop=True)
    obs = plot_df["observed_rate"].to_numpy()
    pred = plot_df["predicted_rate_mean"].to_numpy()
    axD.plot([0, 1], [0, 1], linestyle="--", color="#555555", linewidth=1.1)
    texts = []
    for _, row in plot_df.iterrows():
        species_color = SPECIES_COLORS.get(row["species"], "#6c757d")
        axD.errorbar(
            row["observed_rate"],
            row["predicted_rate_mean"],
            yerr=[[row["predicted_rate_mean"] - row["predicted_rate_hdi_3"]], [row["predicted_rate_hdi_97"] - row["predicted_rate_mean"]]],
            fmt="o",
            color=species_color,
            ecolor=lighten_color(species_color, 0.55),
            elinewidth=1.1,
            markeredgecolor="white",
            markeredgewidth=0.5,
            markersize=5.2,
            capsize=3,
            zorder=3,
        )
        texts.append(axD.text(row["observed_rate"], row["predicted_rate_mean"], species_short(row["species"]), fontsize=6.0, color=species_color))
    axD.set_xlim(-0.02, 1.0)
    axD.set_ylim(-0.02, 1.0)
    axD.set_xlabel("Observed reassortment rate")
    axD.set_ylabel("Posterior predictive mean")
    adjust_text(
        texts,
        x=obs,
        y=pred,
        ax=axD,
        avoid_self=True,
        force_text=(0.8, 0.8),
        force_static=(1.2, 1.2),
        force_pull=(0.15, 0.15),
        expand=(1.15, 1.25),
        arrowprops=dict(arrowstyle="-", color="#888888", lw=0.5),
    )

    fig.savefig(outpath, dpi=600, bbox_inches="tight")
    plt.close(fig)
    return outpath


In [ ]:
fig1 = make_figure1(outpath=str(REPO_ROOT / "results" / "figures" / "figure1_model.png"))
print(fig1)

In [ ]:
def make_figure2(hier_dir=HIER_DIR, dataset_path=DATASET, outpath=None):
    outpath = outpath or FIGURE_SAVE_PATH / "figure2_mechanism_inference.png"
    fx = pd.read_csv(hier_dir / "fixed_effect_comparison.tsv", sep="	")
    species_re = pd.read_csv(hier_dir / "species_random_effects.tsv", sep="	")
    raw = pd.read_csv(dataset_path, sep="	", low_memory=False)
    raw = raw[raw["is_reassortant_final"].notna()].copy()
    raw = raw[raw["target_source"].isin(["sequence_true_label", "isolate_true_label"])].copy()
    raw["target"] = pd.to_numeric(raw["is_reassortant_final"], errors="coerce")

    fig = plt.figure(figsize=(10.6, 7.0), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1.0], width_ratios=[1.28, 1.0])
    axA = fig.add_subplot(gs[:, 0])
    axB = fig.add_subplot(gs[0, 1])
    axC = fig.add_subplot(gs[1, 1])

    panel_label(axA, "A")
    panel_label(axB, "B")
    panel_label(axC, "C")

    forest_model_order = [
        "ecology_interaction_block",
        "molecular_plus_utr3_block",
        "integrated_plus_utr3_block",
        "integrated_tree_sensitivity",
    ]
    forest = fx[fx["model"].isin(forest_model_order)].copy()
    present_terms = list(dict.fromkeys(forest["term"].tolist()))
    preferred_terms = [
        "hosttrait_combine_adult_mass_g_z",
        "hosttrait_combine_litter_size_n_z",
        "hosttrait_combine_gestation_length_d_z",
        "hosttrait_combine_dispersal_km_z",
        "species_feat_hostdiv_faith_pd_z",
        "overlap_opportunity_local_z",
        "arha_prev_local_z",
        "overlap_opportunity_local_z:species_feat_hostdiv_faith_pd_z",
        "species_feat_mean_cramers_v_overall_z",
        "species_feat_prop_significant_overall_z",
        "species_feat_utr5_S_structdist_mean_z",
        "species_feat_utr5_M_structdist_mean_z",
        "species_feat_utr5_L_structdist_mean_z",
        "species_feat_utr3_S_structdist_mean_z",
        "species_feat_utr3_M_structdist_mean_z",
        "species_feat_utr3_L_structdist_mean_z",
        "tree_discordance_score_z",
    ]
    selected_terms = ordered_present(present_terms, preferred_terms)
    forest = forest[forest["term"].isin(selected_terms)].copy()
    forest["term_label"] = forest["term"].map(pretty_term)
    forest["model_label"] = forest["model"].map(BLOCK_DISPLAY).fillna(forest["model"])
    forest["term_label"] = pd.Categorical(forest["term_label"], categories=[pretty_term(t) for t in selected_terms], ordered=True)
    forest["model_label"] = pd.Categorical(forest["model_label"], categories=[BLOCK_DISPLAY.get(m, m) for m in forest_model_order if m in forest["model"].unique()], ordered=True)
    forest = forest.sort_values(["model_label", "term_label"]).reset_index(drop=True)

    model_colors = {
        BLOCK_DISPLAY.get("ecology_interaction_block"): "#2a9d8f",
        BLOCK_DISPLAY.get("molecular_plus_utr3_block"): "#577590",
        BLOCK_DISPLAY.get("integrated_plus_utr3_block"): "#bc4749",
        BLOCK_DISPLAY.get("integrated_tree_sensitivity"): "#8d99ae",
    }
    y = np.arange(len(forest))
    for block_idx, (_, block) in enumerate(forest.groupby("model_label", sort=False)):
        if block_idx % 2 == 0:
            axA.axhspan(block.index.min() - 0.5, block.index.max() + 0.5, color="#fafafa", zorder=0)
    axA.hlines(y, forest["hdi_3%"], forest["hdi_97%"], color=[model_colors[m] for m in forest["model_label"]], linewidth=2)
    axA.scatter(forest["mean"], y, color=[model_colors[m] for m in forest["model_label"]], s=26, zorder=3)
    axA.axvline(0.0, linestyle="--", color="#555555", linewidth=1.0)
    axA.set_yticks(y, forest["term_label"])
    axA.invert_yaxis()
    axA.set_xlabel("Posterior coefficient")
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="", markersize=5.5, markerfacecolor=model_colors[label], markeredgecolor=model_colors[label], label=label)
        for label in forest["model_label"].cat.categories
    ]
    axA.legend(handles=legend_handles, frameon=False, fontsize=6.3, ncol=2, loc="lower left", bbox_to_anchor=(0.0, 1.01, 1.0, 0.08), mode="expand", borderaxespad=0.0, handletextpad=0.35, columnspacing=0.9)

    shrink_models = [m for m in ["species_only", "integrated_plus_utr3_block"] if m in species_re["model"].unique()]
    shrink = species_re[species_re["model"].isin(shrink_models)].copy()
    species_order = shrink[shrink["model"] == shrink_models[0]].sort_values("mean")["species"].tolist()
    ypos = np.arange(len(species_order))
    offsets = {shrink_models[0]: -0.12, shrink_models[-1]: 0.12}
    point_colors = {shrink_models[0]: "#8d99ae", shrink_models[-1]: "#bc4749"}
    legend_labels = {shrink_models[0]: "Species only", shrink_models[-1]: "Integrated + UTR3"}
    for model in shrink_models:
        sub = shrink[shrink["model"] == model].set_index("species").loc[species_order].reset_index()
        yy = ypos + offsets[model]
        axB.hlines(yy, sub["hdi_3%"], sub["hdi_97%"], color=point_colors[model], linewidth=2, alpha=0.88)
        axB.scatter(sub["mean"], yy, color=point_colors[model], s=30, label=legend_labels[model])
    axB.axvline(0.0, linestyle="--", color="#555555", linewidth=1.0)
    axB.set_yticks(ypos, [species_short(s) for s in species_order])
    axB.set_xlabel("Species random effect")
    axB.legend(frameon=False, fontsize=7.2, loc="lower right")

    heat_features = [
        "observed_rate",
        "overlap_opportunity_local",
        "arha_prev_local",
        "species_feat_hostdiv_faith_pd",
        "species_feat_mean_cramers_v_overall",
        "species_feat_prop_significant_overall",
        "species_feat_utr5_S_structdist_mean",
        "species_feat_utr5_M_structdist_mean",
        "species_feat_utr5_L_structdist_mean",
        "species_feat_utr3_S_structdist_mean",
        "species_feat_utr3_M_structdist_mean",
        "species_feat_utr3_L_structdist_mean",
    ]
    heat_features = [c for c in heat_features if c == "observed_rate" or c in raw.columns]
    species_feature = raw.groupby("species").agg(observed_rate=("target", "mean")).reset_index()
    feature_cols = [c for c in heat_features if c != "observed_rate"]
    feature_table = raw.groupby("species")[feature_cols].first().reset_index() if feature_cols else raw[["species"]].drop_duplicates().copy()
    heat = species_feature.merge(feature_table, on="species", how="left").sort_values("observed_rate").reset_index(drop=True)
    heat_mat = heat[heat_features].copy()
    for col in heat_mat.columns:
        if col == "observed_rate":
            continue
        series = pd.to_numeric(heat_mat[col], errors="coerce")
        std = series.std(ddof=0)
        heat_mat[col] = 0.0 if pd.isna(std) or std == 0 else (series - series.mean()) / std
    heat_mat["observed_rate"] = heat["observed_rate"]
    data = heat_mat.to_numpy(dtype=float)
    cmap = mpl.colormaps["RdYlBu_r"].copy()
    cmap.set_bad("#e5e5e5")
    im = axC.imshow(np.ma.masked_invalid(data), aspect="auto", cmap=cmap, vmin=-2.2, vmax=2.2)
    axC.set_yticks(np.arange(len(heat)), [species_short(s) for s in heat["species"]])
    axC.set_xticks(np.arange(len(heat_features)), [HEATMAP_LABELS.get(c, c) for c in heat_features], rotation=38, ha="right")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = heat.iloc[i]["observed_rate"] if heat_features[j] == "observed_rate" else data[i, j]
            label = "" if pd.isna(value) else f"{value:.2f}"
            axC.text(j, i, label, ha="center", va="center", fontsize=5.6, color="#1b1b1b")
    cbar = fig.colorbar(im, ax=axC, fraction=0.046, pad=0.04)
    cbar.set_label("Standardized feature value")

    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return outpath


In [ ]:
fig2 = make_figure2(outpath=str(REPO_ROOT / "results" / "figures" / "figure2_mechanism_inference.png"))
print(fig2)

## Figure 3: Integrated Explanation Architecture

This section mirrors the integrated-model explanation figure used in the manuscript. It now uses tuned `integrated_taxon_mechanism / lightgbm` TreeSHAP contributions from the final defended model set, collapsed into biological modules for visualization. The previous linear L2 version is preserved under `legacy_linear_l2`.


## Figure 4: Lineage Holdout Trees

In [ ]:
def make_figure4(suite_dir=PROMOTED_SUITE_DIR, outpath=None):
    import tempfile
    import baltic as bt
    from Bio import Phylo
    from matplotlib import cm, colors
    from sklearn.metrics import balanced_accuracy_score

    outpath = outpath or FIGURE_SAVE_PATH / "figure4_lineage_tree_holdout.png"
    pred_path = suite_dir / "oof_predictions.tsv"
    manifest_path = suite_dir / "within_species_tree_clade_manifest.tsv"

    pred_df = pd.read_csv(pred_path, sep="	")
    pred_df = pred_df[
        (pred_df["cv_scheme"] == "grouped_within_species_clade")
        & (pred_df["model"] == "xgboost")
        & (pred_df["feature_set"] == "ecology_molecular")
    ].copy()

    threshold_grid = np.linspace(0.02, 0.98, 97)
    fold_thresholds = {}
    for fold in sorted(pred_df["fold_id"].dropna().unique()):
        train = pred_df[pred_df["fold_id"] != fold]
        if train.empty:
            fold_thresholds[fold] = 0.5
            continue
        fold_thresholds[fold] = max(
            threshold_grid,
            key=lambda t: balanced_accuracy_score(train["target"], (train["probability"] >= t).astype(int)),
        )
    pred_df["threshold_balacc_opt"] = pred_df["fold_id"].map(fold_thresholds).fillna(0.5)
    pred_df["predicted_label"] = (pred_df["probability"] >= pred_df["threshold_balacc_opt"]).astype(int)

    manifest_df = pd.read_csv(manifest_path, sep="	")
    manifest_df = manifest_df[manifest_df["status"].eq("assigned")].copy()

    species_order = pred_df.groupby("species").size().sort_values(ascending=False).index.tolist()
    manifest_df = manifest_df.set_index("species").loc[species_order].reset_index()

    pretty_species = {
        "orthohantavirus_andesense": "Andes virus",
        "orthohantavirus_dobravaense": "Dobrava-Belgrade virus",
        "orthohantavirus_hantanense": "Hantaan virus",
        "orthohantavirus_puumalaense": "Puumala virus",
        "orthohantavirus_seoulense": "Seoul virus",
        "orthohantavirus_sinnombreense": "Sin Nombre virus",
        "orthohantavirus_tulaense": "Tula virus",
    }

    prob_bins = [0.0, 0.10, 0.25, 0.40, 0.60, 1.0]
    prob_colors = ["#180f3d", "#5d177e", "#9b2f7f", "#d8576b", "#f7c35b"]
    obs_cmap = colors.ListedColormap(prob_colors, name="obs_bins")
    obs_norm = colors.BoundaryNorm(prob_bins, obs_cmap.N, clip=True)
    pred_cmap = colors.ListedColormap(prob_colors, name="pred_bins")
    pred_norm = colors.BoundaryNorm(prob_bins, pred_cmap.N, clip=True)
    binary_tip_colors = {0: prob_colors[0], 1: prob_colors[-1]}

    panel_data = []
    for _, row in manifest_df.iterrows():
        species = row["species"]
        tree_path = Path(row["tree_path"])
        sub = pred_df[pred_df["species"].eq(species)].copy()
        if sub.empty:
            continue

        tip_target = sub.set_index("tip_name")["target"].to_dict()
        tip_pred = sub.set_index("tip_name")["predicted_label"].to_dict()
        tip_clade = sub.set_index("tip_name")["within_species_clade"].to_dict()
        clade_summary = (
            sub.groupby("within_species_clade", dropna=False)
            .agg(obs_rate=("target", "mean"), pred_rate=("probability", "mean"), pred_call_rate=("predicted_label", "mean"), n=("tip_name", "size"))
            .reset_index()
        )
        evaluated_tips = set(sub["tip_name"].astype(str))

        pruned_tree = Phylo.read(str(tree_path), "newick")
        for terminal in list(pruned_tree.get_terminals()):
            if terminal.name not in evaluated_tips:
                pruned_tree.prune(terminal)

        with tempfile.NamedTemporaryFile(mode="w", suffix=".nwk", delete=False) as handle:
            Phylo.write(pruned_tree, handle.name, "newick")
            pruned_tree_path = handle.name

        tree = bt.loadNewick(pruned_tree_path, absoluteTime=False)
        tree.traverse_tree()
        tree.sortBranches()
        leaves = [k for k in tree.Objects if k.branchType == "leaf"]
        ys = [k.y for k in leaves]
        max_x = max(k.x for k in tree.Objects)
        leaf_y = {k.name: k.y for k in leaves if k.name in evaluated_tips}

        clade_bounds = []
        for _, crow in clade_summary.iterrows():
            clade = crow["within_species_clade"]
            names = [name for name, label in tip_clade.items() if label == clade and name in leaf_y]
            if not names:
                continue
            yvals = [leaf_y[name] for name in names]
            clade_bounds.append(
                {
                    "clade": clade,
                    "ymin": min(yvals) - 0.45,
                    "ymax": max(yvals) + 0.45,
                    "ymid": float(np.mean(yvals)),
                    "obs_rate": float(crow["obs_rate"]),
                    "pred_rate": float(crow["pred_rate"]),
                    "pred_call_rate": float(crow["pred_call_rate"]),
                    "n": int(crow["n"]),
                }
            )
        clade_bounds = sorted(clade_bounds, key=lambda d: d["ymid"])
        clade_metric = {d["clade"]: d for d in clade_bounds}

        panel_data.append(
            {
                "species": species,
                "title": pretty_species.get(species, SPECIES_LABELS.get(species, species)),
                "n": len(sub),
                "tree": tree,
                "ys": ys,
                "max_x": max_x,
                "tip_target": tip_target,
                "tip_pred": tip_pred,
                "tip_clade": tip_clade,
                "clade_bounds": clade_bounds,
                "clade_metric": clade_metric,
            }
        )

    fig = plt.figure(figsize=(15.2, 13.8), constrained_layout=False)
    outer = fig.add_gridspec(2, 1, left=0.04, right=0.965, top=0.97, bottom=0.06, hspace=0.18)

    panel_specs = [
        ("A", obs_cmap, obs_norm, "obs_rate", "Observed clade reassortment proportion"),
        ("B", pred_cmap, pred_norm, "pred_rate", "Held-out predicted clade probability"),
    ]

    for panel_idx, (letter, cmap, norm, metric_key, legend_label) in enumerate(panel_specs):
        n_cols = 3
        n_rows = int(np.ceil(len(panel_data) / n_cols))
        gs = outer[panel_idx].subgridspec(n_rows, n_cols + 1, width_ratios=[1, 1, 1, 0.08], wspace=0.14, hspace=0.24)
        tree_axes = [fig.add_subplot(gs[i, j]) for i in range(n_rows) for j in range(n_cols)]
        cax = fig.add_subplot(gs[:, n_cols])

        for ax_idx, ax in enumerate(tree_axes):
            if ax_idx >= len(panel_data):
                ax.axis("off")
                continue
            info = panel_data[ax_idx]
            tree = info["tree"]
            ys = info["ys"]
            max_x = info["max_x"]
            clade_metric = info["clade_metric"]

            def branch_colour(k):
                if getattr(k, 'branchType', None) == 'leaf':
                    clade = info["tip_clade"].get(k.name)
                else:
                    descendant_tips = [name for name in getattr(k, 'leaves', set()) if name in info["tip_clade"]]
                    clades = {info["tip_clade"][name] for name in descendant_tips}
                    clade = next(iter(clades)) if len(clades) == 1 else None
                if clade is None or clade not in clade_metric:
                    return "#9a9a9a"
                return cmap(norm(clade_metric[clade][metric_key]))

            tree.plotTree(ax, width=lambda k: 1.15 if k.branchType == "leaf" else 1.0, colour=branch_colour)

            if metric_key == "obs_rate":
                def tip_colour(k):
                    if k.branchType != "leaf":
                        return "#8c8c8c"
                    val = info["tip_target"].get(k.name, np.nan)
                    if np.isnan(val):
                        return "#c9c9c9"
                    return binary_tip_colors[int(val)]
            else:
                def tip_colour(k):
                    if k.branchType != "leaf":
                        return "#8c8c8c"
                    val = info["tip_pred"].get(k.name, np.nan)
                    if np.isnan(val):
                        return "#c9c9c9"
                    return binary_tip_colors[int(val)]

            tree.plotPoints(ax, size=lambda k: 12 if k.branchType == "leaf" else 0, colour=tip_colour, outline=False, zorder=4)

            label_x = max_x * 1.03
            for idx, b in enumerate(info["clade_bounds"], start=1):
                ax.text(label_x, b["ymid"], f"L{idx}", va="center", ha="left", fontsize=6.3, color="#6b6b6b")

            ax.text(0.01, 1.01, info["title"], transform=ax.transAxes, ha="left", va="bottom", fontsize=10.0, fontweight="bold", color=SPECIES_COLORS.get(info["species"], "#333333"))
            ax.text(0.995, 1.01, f"n={info['n']}", transform=ax.transAxes, ha="right", va="bottom", fontsize=6.2, color="#666666")
            ax.set_ylim(min(ys) - 1, max(ys) + 1)
            ax.set_xlim(-0.002, label_x + max_x * 0.05)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

            if ax_idx == 0:
                panel_label(ax, f"{letter})", x=-0.06, y=1.02)

        sm = cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, cax=cax)
        mids = [(a + b) / 2 for a, b in zip(prob_bins[:-1], prob_bins[1:])]
        cbar.set_ticks(mids)
        cbar.set_ticklabels(["0-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60", ">0.60"])
        cbar.set_label(legend_label, fontsize=7)
        cbar.ax.tick_params(labelsize=6)

    fig.savefig(outpath, dpi=400, bbox_inches="tight")
    plt.close(fig)
    return outpath



In [ ]:
fig4 = make_figure4(outpath=FIGURE_SAVE_PATH / "figure4_lineage_tree_holdout.png")

## Figure 3B: Integrated Explanation Architecture, Module-first Revision

In [ ]:
def make_figure3_vb(fig3_data_dir=FIG3_DATA_DIR, outpath=None):
    from scipy.cluster.hierarchy import dendrogram, linkage

    outpath = outpath or FIGURE_SAVE_PATH / "figure3_integrated_explanation_architecture_vb.png"
    module_df = pd.read_csv(fig3_data_dir / "integrated_module_contributions.tsv", sep="\t")
    rank_df = pd.read_csv(fig3_data_dir / "integrated_feature_rank_summary.tsv", sep="\t")
    shift_df = pd.read_csv(fig3_data_dir / "integrated_feature_rank_shift.tsv", sep="\t")
    cluster_df = pd.read_csv(fig3_data_dir / "integrated_top_feature_clusters.tsv", sep="\t")

    module_candidates = [c for c in module_df.columns if c not in {"tip_name", "species", "target", "probability"}]
    preferred_module_order = [
        "Species identity",
        "Host life history",
        "Host phylogenetic diversity",
        "Overlap opportunity",
        "Local host prevalence",
        "Geographic / sampling context",
        "LD burden",
        "Significant LD fraction",
        "UTR5 S structure",
        "UTR5 M structure",
        "UTR5 L structure",
        "UTR3 S structure",
        "UTR3 M structure",
        "UTR3 L structure",
        "Tree discordance",
        "Controls / sampling",
    ]
    module_order = ordered_present(module_candidates, preferred_module_order)

    def clean_family_label(name):
        return str(name).replace(" / ", "\n")

    def clean_feat_label(label, width=20, max_lines=2):
        label = str(label)
        replacements = {
            "hosttrait combine ": "",
            "species feat ": "",
            "ld site ": "LD site ",
            "utr5 ": "UTR5 ",
            "utr3 ": "UTR3 ",
            " tree discordance score": "tree discordance",
            "mean cramers v overall": "mean LD burden",
            "significant fraction": "significant LD fraction",
            "local host prevalence": "local host prevalence",
        }
        for old_text, new_text in replacements.items():
            label = label.replace(old_text, new_text)
        label = label.replace("_", " ")
        parts = label.split()
        lines_out = []
        cur = []
        n = 0
        for part in parts:
            extra = len(part) + (1 if cur else 0)
            if cur and n + extra > width:
                lines_out.append(" ".join(cur))
                cur = [part]
                n = len(part)
                if len(lines_out) >= max_lines:
                    break
            else:
                cur.append(part)
                n += extra
        if len(lines_out) < max_lines and cur:
            lines_out.append(" ".join(cur))
        return "\n".join(lines_out[:max_lines])

    fig = plt.figure(figsize=(12.8, 8.7), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.35, 1.0], width_ratios=[1.05, 1.0, 1.08])
    top = gs[0, :].subgridspec(4, 2, height_ratios=[0.58, 0.12, 1.0, 0.34], wspace=0.02, hspace=0.0)

    pos_mask = module_df["target"].to_numpy() == 1
    neg_mask = ~pos_mask
    groups = [("Reassortant", pos_mask, "#c44e52"), ("Non-reassortant", neg_mask, "#4c72b0")]

    heat_vals = module_df[module_order].to_numpy(dtype=float)
    clip = np.nanquantile(np.abs(heat_vals), 0.98)
    clip = 1.0 if not np.isfinite(clip) or clip == 0 else clip
    cmap = mpl.colors.LinearSegmentedColormap.from_list("white_centered", ["#7ea6d8", "#ffffff", "#e0a1a9"], N=256)

    for col_idx, (group_label, mask, accent) in enumerate(groups):
        sub_df = module_df.loc[mask].copy().reset_index(drop=True)
        sub = sub_df[module_order].to_numpy(dtype=float)
        if len(sub_df) > 2:
            Z = linkage(sub, method="average", metric="euclidean")
            order = dendrogram(Z, no_plot=True)["leaves"]
            sub_df = sub_df.iloc[order].reset_index(drop=True)
            sub = sub_df[module_order].to_numpy(dtype=float)
        else:
            Z = None

        den_ax = fig.add_subplot(top[0, col_idx])
        strip_ax = fig.add_subplot(top[1, col_idx], sharex=den_ax)
        heat_ax = fig.add_subplot(top[2, col_idx], sharex=den_ax)
        prob_ax = fig.add_subplot(top[3, col_idx], sharex=den_ax)

        if Z is not None:
            dendrogram(
                Z,
                ax=den_ax,
                no_labels=True,
                color_threshold=0,
                above_threshold_color="#7f7f7f",
                link_color_func=lambda _: "#7f7f7f",
            )
            for coll in den_ax.collections:
                coll.set_clip_on(False)
            den_ax.margins(y=0.2)
            ymin, ymax = den_ax.get_ylim()
            den_ax.set_ylim(-0.06 * ymax if ymax > 0 else ymin, ymax * 1.15 if ymax > 0 else ymax)
        den_ax.text(0.02, 1.02, group_label, transform=den_ax.transAxes, ha="left", va="bottom", fontsize=7.2, color=accent)
        den_ax.set_xticks([])
        den_ax.set_yticks([])
        for spine in den_ax.spines.values():
            spine.set_visible(False)
        if col_idx == 0:
            panel_label(den_ax, "A")

        strip_colors = np.asarray([mpl.colors.to_rgb(SPECIES_COLORS.get(s, "#bbbbbb")) for s in sub_df["species"]])[np.newaxis, :, :]
        x_extent = (-0.5, max(len(sub_df) - 0.5, 0.5)) if Z is None else (0, len(sub_df) * 10)
        strip_ax.imshow(strip_colors, aspect="auto", interpolation="nearest", extent=[x_extent[0], x_extent[1], 0, 1])
        strip_ax.set_yticks([])
        strip_ax.set_xticks([])
        for spine in strip_ax.spines.values():
            spine.set_visible(False)

        heat_ax.set_facecolor("white")
        heat_ax.imshow(
            sub.T,
            aspect="auto",
            cmap=cmap,
            vmin=-clip,
            vmax=clip,
            interpolation="nearest",
            extent=[x_extent[0], x_extent[1], len(module_order) - 0.5, -0.5],
        )
        heat_ax.set_yticks(np.arange(len(module_order)), module_order if col_idx == 0 else [])
        heat_ax.set_xticks([])
        for spine in heat_ax.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#444444")

        probs = sub_df["probability"].to_numpy(dtype=float)
        if Z is None:
            bar_x = np.arange(len(probs))
            bar_w = 1.0
        else:
            bar_x = np.arange(len(probs)) * 10 + 5
            bar_w = 8.5
        prob_ax.bar(bar_x, probs, color=accent, width=bar_w)
        prob_ax.axhline(module_df["target"].mean(), color="#666666", linestyle=(0, (4, 4)), linewidth=1.0)
        prob_ax.set_ylim(0, 1)
        prob_ax.set_yticks([0, 0.5, 1.0] if col_idx == 0 else [])
        prob_ax.set_ylabel("Predicted\nprobability" if col_idx == 0 else "")
        prob_ax.set_xticks([])
        den_ax.set_xlim(x_extent[0], x_extent[1])
        if col_idx == 0:
            prob_ax.set_xlabel("Isolates (sorted by explanation similarity)")

    axB = fig.add_subplot(gs[1, 0])
    axC = fig.add_subplot(gs[1, 1])
    axD = fig.add_subplot(gs[1, 2])
    panel_label(axB, "B")
    panel_label(axC, "C")
    panel_label(axD, "D")

    families = [fam for fam in FAMILY_COLORS if fam in rank_df["family"].values and fam != "Controls / sampling"]
    values = [rank_df.loc[rank_df["family"] == fam, "mean_rank"].to_numpy() for fam in families]
    bp = axB.boxplot(
        values,
        vert=False,
        widths=0.62,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "#333333", "linewidth": 1.0},
        whiskerprops={"color": "#777777", "linewidth": 0.8},
        capprops={"color": "#777777", "linewidth": 0.8},
        boxprops={"edgecolor": "#666666", "linewidth": 0.8},
    )
    for patch, fam in zip(bp["boxes"], families):
        patch.set_facecolor(mpl.colors.to_rgba(FAMILY_COLORS[fam], 0.20))
        patch.set_edgecolor(FAMILY_COLORS[fam])
    half_span = 0.42
    for idx, fam in enumerate(families, start=1):
        vals = np.sort(values[idx - 1])
        for val in vals:
            axB.vlines(val, idx - half_span, idx + half_span, color="#cfcfcf", linewidth=0.9, zorder=0, alpha=0.9)
    axB.set_yticks(np.arange(1, len(families) + 1), [clean_family_label(fam) for fam in families])
    axB.set_xlabel("Mean feature rank")
    axB.invert_xaxis()

    plot_df = (
        shift_df[
            ~shift_df["feature_label"].str.startswith("Missingness:")
            & shift_df["family"].ne("Controls / sampling")
        ]
        .head(10)
        .sort_values("abs_rank_shift", ascending=True)
        .reset_index(drop=True)
    )
    y = np.arange(len(plot_df))
    for idx, row in plot_df.iterrows():
        color = FAMILY_COLORS.get(row["family"], "#777777")
        axC.plot([row["non_reassortant_mean_rank"], row["reassortant_mean_rank"]], [idx, idx], color=color, alpha=0.75, linewidth=1.2)
        axC.scatter(row["non_reassortant_mean_rank"], idx, color="#777777", s=14, zorder=3)
        axC.scatter(row["reassortant_mean_rank"], idx, color=color, s=18, zorder=3)
    axC.set_yticks(y, [clean_feat_label(x, width=22, max_lines=2) for x in plot_df["feature_label"]])
    axC.tick_params(axis="y", labelsize=6.0)
    axC.set_xlabel("Mean feature rank")
    axC.invert_xaxis()

    plot_clusters = (
        cluster_df[cluster_df["family_Controls / sampling"] < cluster_df["combined_effect"]]
        .sort_values("combined_effect", ascending=True)
        .tail(14)
        .reset_index(drop=True)
    )
    y2 = np.arange(len(plot_clusters))
    left = np.zeros(len(plot_clusters))
    for family, color in FAMILY_COLORS.items():
        if family == "Controls / sampling":
            continue
        col = f"family_{family}"
        if col not in plot_clusters.columns:
            continue
        vals = plot_clusters[col].to_numpy(dtype=float)
        if np.allclose(vals, 0):
            continue
        axD.barh(y2, vals, left=left, color=color, edgecolor="none", label=family)
        left = left + vals
    axD.set_yticks(y2, plot_clusters["rank"].astype(str))
    axD.set_xlabel("Combined mean absolute contribution")
    axD.set_ylabel("Feature cluster")
    xmax = float(plot_clusters["combined_effect"].max()) * 1.48
    axD.set_xlim(0, xmax)
    for yi, (_, row) in zip(y2, plot_clusters.iterrows()):
        axD.text(
            row["combined_effect"] + xmax * 0.02,
            yi,
            clean_feat_label(row["exemplar_label"], width=18, max_lines=2),
            va="center",
            fontsize=5.4,
        )
    axD.legend(frameon=False, fontsize=5.0, loc="upper left", bbox_to_anchor=(0.0, -0.18), ncol=2, columnspacing=0.7, handlelength=1.0)

    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return outpath


In [ ]:
fig3b = make_figure3_vb(outpath=FIGURE_SAVE_PATH / "figure3_integrated_explanation_architecture_vb.png")
print(fig3b)

## Figure 5: LD Interaction-Only Sensitivity

## Figure 6: Species-Specific Block Response Surfaces

Species-varying-slope response surfaces over ecology and molecular block scores, using the strict expanded dataset and the integrated decomposition as the block-score source. This reuses the Figure 6 visual language but splits the response landscape by species.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import subprocess, sys

repo_root = REPO_ROOT
figure_dir = repo_root / 'notebooks' / 'figures'
module_table = HIER_DIR / 'figures' / 'manuscript_utr3_integrated' / 'integrated_module_contributions.tsv'
subprocess.run([
    sys.executable,
    str(repo_root / 'scripts' / 'modeling' / 'fit_species_varying_slope_block_model.py'),
    '--module-table', str(module_table),
    '--output-dir', str(repo_root / 'results' / 'modeling' / 'species_varying_slope_block_model'),
    '--figure-path', str(figure_dir / 'figure6_species_block_surface_facets.png'),
], check=True, cwd=repo_root)

fig6_species = figure_dir / 'figure6_species_block_surface_facets.png'
display(Image(filename=str(fig6_species), width=1100))


## Figure 7: Exploratory Ecology × Molecular Interaction Surfaces

Exploratory 2D surfaces over selected ecology × molecular or ecology × phylogenetic pairs. These are descriptive visual diagnostics from the integrated decomposition and do not replace the formal hierarchical interaction comparison.

In [ ]:
from scipy.interpolate import RBFInterpolator
from scipy.linalg import LinAlgError


def make_figure7_ecology_molecular_utr(fig3_data_dir=FIG3_DATA_DIR, outpath=None, grid_size=60):
    outpath = outpath or FIGURE_SAVE_PATH / 'ecology_molecular_interaction_surfaces.png'
    df = pd.read_csv(fig3_data_dir / 'integrated_module_contributions.tsv', sep='	').copy()
    utr_cols = [c for c in df.columns if c.startswith('UTR')]
    df['UTR compatibility'] = df[utr_cols].sum(axis=1)

    pair_specs = [
        ('Overlap opportunity', 'LD burden'),
        ('Overlap opportunity', 'Significant LD fraction'),
        ('Local host prevalence', 'LD burden'),
        ('Local host prevalence', 'Significant LD fraction'),
        ('Overlap opportunity', 'UTR compatibility'),
        ('Local host prevalence', 'UTR compatibility'),
    ]
    pair_specs = [pair for pair in pair_specs if pair[0] in df.columns and pair[1] in df.columns]

    def pretty_label(label):
        return f'{label} contribution (logit)'

    def smooth_surface(x, y, z):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        z = np.asarray(z, dtype=float)
        x_lo, x_hi = float(np.min(x)), float(np.max(x))
        y_lo, y_hi = float(np.min(y)), float(np.max(y))
        x_pad = max((x_hi - x_lo) * 0.03, 1e-6)
        y_pad = max((y_hi - y_lo) * 0.03, 1e-6)
        grid_x, grid_y = np.meshgrid(np.linspace(x_lo - x_pad, x_hi + x_pad, grid_size), np.linspace(y_lo - y_pad, y_hi + y_pad, grid_size))
        try:
            rbf = RBFInterpolator(np.column_stack([x, y]), z, smoothing=0.45, kernel='thin_plate_spline')
            grid_z = rbf(np.column_stack([grid_x.ravel(), grid_y.ravel()])).reshape(grid_x.shape)
        except (ValueError, LinAlgError, np.linalg.LinAlgError):
            design = np.column_stack([np.ones_like(x), x, y])
            coef, *_ = np.linalg.lstsq(design, z, rcond=None)
            grid_z = coef[0] + coef[1] * grid_x + coef[2] * grid_y
        return grid_x, grid_y, np.clip(grid_z, 0.0, 1.0)

    n_panels = len(pair_specs)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(11.0, 6.8), constrained_layout=False)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.96, bottom=0.09, wspace=0.24, hspace=0.28)
    axes = np.atleast_1d(axes).ravel()
    cmap = mpl.colormaps['viridis']
    panel_letters = list('ABCDEF')

    for ax, (xcol, ycol), letter in zip(axes, pair_specs, panel_letters):
        grid_x, grid_y, grid_z = smooth_surface(df[xcol], df[ycol], df['probability'])
        levels = np.linspace(0.0, 1.0, 11)
        cf = ax.contourf(grid_x, grid_y, grid_z, levels=levels, cmap=cmap, vmin=0.0, vmax=1.0)
        ax.contour(grid_x, grid_y, grid_z, levels=np.linspace(0.1, 0.9, 5), colors='white', linewidths=0.5, alpha=0.65)
        neg = df['target'].to_numpy(dtype=int) == 0
        pos = ~neg
        ax.scatter(df.loc[neg, xcol], df.loc[neg, ycol], s=10, c='#7f7f7f', alpha=0.32, edgecolors='none')
        ax.scatter(df.loc[pos, xcol], df.loc[pos, ycol], s=12, c='#c44e52', alpha=0.65, edgecolors='white', linewidths=0.2)
        panel_label(ax, f'{letter})', x=-0.08, y=1.02)
        ax.set_xlabel(pretty_label(xcol))
        ax.set_ylabel(pretty_label(ycol))
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    for ax in axes[n_panels:]:
        ax.axis('off')

    cbar = fig.colorbar(cf, ax=axes.tolist(), shrink=0.92, pad=0.02)
    cbar.set_label('Smoothed model probability')
    fig.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return outpath

fig7 = make_figure7_ecology_molecular_utr(outpath=FIGURE_SAVE_PATH / 'ecology_molecular_interaction_surfaces.png')
from IPython.display import Image, display
display(Image(filename=str(fig7), width=1000))


In [ ]:
# Figure 5 (Version B): LD interaction-only sensitivity for the promoted models

def make_figure5_ld_sensitivity_v2(
    species_full=REPO_ROOT / "results" / "modeling" / "final_species_suite",
    species_interaction=REPO_ROOT / "results" / "modeling" / "final_species_suite_ld_interaction_only",
    within_full=REPO_ROOT / "results" / "modeling" / "final_within_species_suite",
    within_interaction=REPO_ROOT / "results" / "modeling" / "final_within_species_suite_ld_interaction_only",
    repeat_full=REPO_ROOT / "results" / "modeling" / "final_repeated_stratified_suite",
    repeat_interaction=REPO_ROOT / "results" / "modeling" / "final_repeated_stratified_suite_ld_interaction_only",
    outpath=None,
):
    outpath = outpath or FIGURE_SAVE_PATH / "figure5_ld_interaction_sensitivity_v2.png"

    configs = [
        ("Species-only RF", species_full, species_interaction),
        ("Within-species XGBoost", within_full, within_interaction),
        ("Repeated-stratified LightGBM", repeat_full, repeat_interaction),
    ]
    rows = []
    for label, full_dir, int_dir in configs:
        full = pd.read_csv(full_dir / "summary.tsv", sep="	").iloc[0]
        inter = pd.read_csv(int_dir / "summary.tsv", sep="	").iloc[0]
        rows.append(
            {
                "model_label": label,
                "full_ap_mean": float(full["average_precision_mean"]),
                "interaction_ap_mean": float(inter["average_precision_mean"]),
                "full_auc_mean": float(full["roc_auc_mean"]),
                "interaction_auc_mean": float(inter["roc_auc_mean"]),
                "full_oof_ap": float(full["oof_average_precision"]),
                "interaction_oof_ap": float(inter["oof_average_precision"]),
                "full_oof_auc": float(full["oof_roc_auc"]),
                "interaction_oof_auc": float(inter["oof_roc_auc"]),
            }
        )
    comp = pd.DataFrame(rows)
    comp["decision"] = np.where(
        comp["interaction_oof_auc"] > comp["full_oof_auc"] + 1e-6,
        "keep interaction-only",
        np.where(
            np.isclose(comp["interaction_oof_auc"], comp["full_oof_auc"]) & np.isclose(comp["interaction_oof_ap"], comp["full_oof_ap"]),
            "neutral",
            "keep full LD",
        ),
    )

    fig, axes = plt.subplots(2, 2, figsize=(10.4, 6.8), constrained_layout=True)
    axes = axes.ravel()
    metric_specs = [
        ("full_ap_mean", "interaction_ap_mean", "Mean average precision", "A"),
        ("full_auc_mean", "interaction_auc_mean", "Mean ROC AUC", "B"),
        ("full_oof_ap", "interaction_oof_ap", "Out-of-fold average precision", "C"),
        ("full_oof_auc", "interaction_oof_auc", "Out-of-fold ROC AUC", "D"),
    ]
    y = np.arange(len(comp))
    base_color = "#9a9a9a"
    interaction_color = "#d6a6a6"

    for ax, (full_col, int_col, xlabel, panel) in zip(axes, metric_specs):
        panel_label(ax, panel)
        for idx, row in comp.iterrows():
            ax.plot([row[full_col], row[int_col]], [idx, idx], color="#bdbdbd", lw=1.4, zorder=1)
            ax.scatter(row[full_col], idx, s=34, color=base_color, edgecolor="white", linewidth=0.4, zorder=3)
            ax.scatter(row[int_col], idx, s=34, color=interaction_color, edgecolor="white", linewidth=0.4, zorder=3)
            delta = row[int_col] - row[full_col]
            ax.text(max(row[full_col], row[int_col]) + 0.003, idx, f"{delta:+.3f}", va="center", fontsize=6.5, color="#666666")
        ax.set_yticks(y, comp["model_label"])
        ax.set_xlabel(xlabel)
        ax.grid(axis="x", color="#e5e5e5", lw=0.6)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].legend(
        handles=[
            Line2D([0], [0], marker="o", color="none", markerfacecolor=base_color, markeredgecolor="white", markeredgewidth=0.4, markersize=5.5, label="Full LD"),
            Line2D([0], [0], marker="o", color="none", markerfacecolor=interaction_color, markeredgecolor="white", markeredgewidth=0.4, markersize=5.5, label="Interaction-only LD"),
        ],
        frameon=False,
        fontsize=6.7,
        loc="lower right",
    )

    decision_ax = axes[3]
    xmax = decision_ax.get_xlim()[1]
    for idx, row in comp.iterrows():
        decision_ax.text(xmax, idx + 0.28, row["decision"], ha="right", va="bottom", fontsize=6.1, color="#555555")

    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    return outpath

fig5_v2 = make_figure5_ld_sensitivity_v2(outpath=FIGURE_SAVE_PATH / "figure5_ld_interaction_sensitivity_v2.png")
fig5_v2
